In [2]:
!pip install -q unsloth trl accelerate evaluate jsonlines sentencepiece


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [3]:

import unsloth
from unsloth import FastLanguageModel

import torch
import json
import random
import gc
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, pipeline
from peft import PeftModel
from trl import SFTTrainer, SFTConfig
import evaluate
import nltk
nltk.download('punkt', quiet=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [39]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

MODEL_NAME = "Qwen/Qwen2.5-3B"

MAX_TOKENS_TRAIN = 300
TARGET_TRAIN_SIZE = 2000
TASK_TYPE = "question-answering"

INFERENCE_MAX_LENGTH = 1024
MAX_NEW_TOKENS = 10

In [8]:
def filter_fn(example):
    if example["task_type"] != TASK_TYPE:
        return False
    length = len(tokenizer(example["inputs"] + " " + example["targets"])["input_ids"])
    return length < MAX_TOKENS

print("Загрузка русского сплита...")
dataset = load_dataset(
    "CohereLabs/aya_collection_language_split",
    "russian",
    split="train",
    streaming=True
)

Загрузка русского сплита...


README.md: 0.00B [00:00, ?B/s]

In [9]:
print(f"Фильтрация: task_type='{TASK_TYPE}', длина < {MAX_TOKENS} токенов...")
filtered_examples = []
for i, example in enumerate(dataset):
    if filter_fn(example):
        filtered_examples.append(example)
        if len(filtered_examples) >= TARGET_SIZE:
            break
    if i % 50000 == 0 and i > 0:
        print(f"Обработано {i} примеров, найдено {len(filtered_examples)} подходящих...")

print(f" Собрано {len(filtered_examples)} примеров для обучения")

Фильтрация: task_type='question-answering', длина < 300 токенов...
 Собрано 3000 примеров для обучения


In [10]:
train_data = Dataset.from_list(filtered_examples)

def format_example(example):
    return {"text": f"{example['inputs'].strip()}\nОтвет: {example['targets'].strip()}"}

formatted_train = train_data.map(format_example, remove_columns=train_data.column_names)
print(f"Пример промпта:\n{formatted_train[0]['text'][:200]}...")

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Пример промпта:
"Эффект поля" - это ярлык, используемый для описания типа? Учитывая предыдущий вопрос, напишите контекст, содержащий ответ. Это может быть от 1 до 20 предложений. Контекст:
Ответ: Существует два типа ...


In [23]:

max_seq_length = 300

print("Загрузка модели с QLoRA...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    load_in_16bit=False,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=RANDOM_SEED,
    max_seq_length=max_seq_length,
)

model.print_trainable_parameters()

Загрузка модели с QLoRA...
==((====))==  Unsloth 2026.4.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383


In [24]:

training_args = SFTConfig(
    output_dir="./qwen25_finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    warmup_steps=50,
    logging_steps=50,
    save_strategy="epoch",
    optim="paged_adamw_32bit",
    seed=RANDOM_SEED,
    max_seq_length=max_seq_length,
    report_to="none",
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_train,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
)

print("Начало обучения...")
trainer.train()

print("Сохранение модели...")
trainer.save_model("./qwen25_finetuned_final")
tokenizer.save_pretrained("./qwen25_finetuned_final")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Начало обучения...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 3 | Total steps = 1,125
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 7,372,800 of 3,093,311,488 (0.24% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
50,1.798882
100,1.519594
150,1.497740
200,1.463432
250,1.459595
300,1.431183
350,1.424873
400,1.357861
450,1.312490
500,1.272477


Сохранение модели...
Обучение завершено, модель сохранена в ./qwen25_finetuned_final


In [26]:

def format_passage_text(raw_text):
    """Улучшает форматирование: '(1) ... (2) ...' → '1. ...\n2. ...'"""
    return re.sub(r'\((\d+)\)\s*', r'\1.\n', raw_text).strip()

def create_muserc_prompt(text, question, candidate):
    """Формирует промпт в стиле, совместимом с обученной моделью"""
    formatted_text = format_passage_text(text)
    return f"""Контекст:
{formatted_text}

Вопрос: {question}
Утверждение: {candidate}
Верно ли это утверждение? Ответь только "да" или "нет"."""

def prepare_muserc_eval(filepath, tokenizer, target_size=150, max_tokens=500, seed=42):
    """
    Подготавливает ровно target_size коротких примеров из MuSeRC.

    Параметры:
    ----------
    filepath : str — путь к val.jsonl
    tokenizer — токенизатор модели (для подсчёта длины)
    target_size : int — сколько примеров вернуть (по умолчанию 150)
    max_tokens : int — максимальная длина промпта в токенах (по умолчанию 500)
    seed : int — для воспроизводимости
    """
    eval_examples = []
    with jsonlines.open(filepath) as reader:
        for obj in reader:
            eval_examples.append(obj)

    random.seed(seed)
    random.shuffle(eval_examples)

    eval_formatted = []

    for ex in eval_examples:
        if len(eval_formatted) >= target_size:
            break

        passage = ex["passage"]
        passage_text = passage["text"]
        questions = passage["questions"]

        for question in questions:
            if len(eval_formatted) >= target_size:
                break
            for candidate in question["answers"]:
                if len(eval_formatted) >= target_size:
                    break

                prompt = create_muserc_prompt(
                    text=passage_text,
                    question=question["question"],
                    candidate=candidate["text"]
                )

                prompt_length = len(tokenizer.encode(prompt))

                if prompt_length <= max_tokens:
                    eval_formatted.append({
                        "prompt": prompt,
                        "label": "да" if candidate["label"] == 1 else "нет",
                        "example_id": ex["idx"],
                        "question": question["question"],
                        "candidate": candidate["text"],
                        "answer_idx": candidate["idx"],
                        "prompt_length": prompt_length,
                    })

    return eval_formatted


print(" Фильтрация: максимум 500 токенов, цель — 150 примеров...")
eval_formatted = prepare_muserc_eval(
    filepath="val.jsonl",
    tokenizer=tokenizer,
    target_size=150,
    max_tokens=500,
    seed=42
)

print(f" Готово {len(eval_formatted)} коротких примеров для оценки")

if eval_formatted:
    prompt_lengths = [item["prompt_length"] for item in eval_formatted]
    print(f"\n Статистика длин промптов:")
    print(f"  Мин: {min(prompt_lengths)}, Макс: {max(prompt_lengths)}, Среднее: {sum(prompt_lengths)//len(prompt_lengths)}")
    print(f"  Все ≤ 500 токенов: {all(l <= 500 for l in prompt_lengths)}")

with open("eval_prompts_short.json", "w", encoding="utf-8") as f:
    json.dump(eval_formatted, f, ensure_ascii=False, indent=2)
print(" Сохранено в eval_prompts_short.json")

if eval_formatted:
    print(f"\n Пример промпта (длина: {eval_formatted[0]['prompt_length']} токенов):")
    print(eval_formatted[0]["prompt"][:400] + "...")
    print(f" Ожидаемый ответ: {eval_formatted[0]['label']}")

 Фильтрация: максимум 500 токенов, цель — 150 примеров...
 Готово 150 коротких примеров для оценки

 Статистика длин промптов:
  Мин: 420, Макс: 500, Среднее: 462
  Все ≤ 500 токенов: True
 Сохранено в eval_prompts_short.json

 Пример промпта (длина: 428 токенов):
Контекст:
1.
Это было давным – давно, когда многое на нашем свете было по–другому. 2.
Реки текли в другую сторону, горы были в другом месте, и тигры были совсем другие. 3.
У них были такие же когти , такие же зубы, но вот шкура была совсем не такая. 4.
Шкура у тигров была яркой, огненнено-рыжей, одной из самых красивых шкур среди всех зверей. 5.
Но вот толку от такой красивой шкуры было очень мало...
 Ожидаемый ответ: да


In [35]:
def extract_answer(text):
    """Извлекает 'да' или 'нет' из ответа модели"""
    if "Ответ:" in text:
        response = text.split("Ответ:")[-1].strip().lower()
    else:
        response = text.strip().lower()
    if "да" in response and "нет" not in response:
        return "да"
    elif "нет" in response:
        return "нет"
    return (text)

In [29]:
def get_prediction(pipe, prompt, max_new_tokens=MAX_NEW_TOKENS):
    """Генерирует ответ и извлекает да/нет"""
    output = pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"]
    generated = output[len(prompt):]
    return extract_answer(generated)

print("\nЗагрузка base модели для инференса...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=INFERENCE_MAX_LENGTH,
    load_in_4bit=True
)

base_pipe = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=base_tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    pad_token_id=base_tokenizer.eos_token_id,
)

print(" Генерация предсказаний base...")
base_preds = []
for i, item in enumerate(eval_formatted):
    pred = get_prediction(base_pipe, item["prompt"])
    base_preds.append(pred)
    if (i + 1) % 50 == 0:
        print(f"  Обработано {i+1}/{len(eval_formatted)}...")
print(f" Base модель: {len(base_preds)} предсказаний готово")

del base_model, base_pipe
gc.collect()
torch.cuda.empty_cache()


Загрузка base модели для инференса...
==((====))==  Unsloth 2026.4.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Генерация предсказаний base...


...

  Обработано 50/150...


Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  Обработано 100/150...


Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  Обработано 150/150...
 Base модель: 150 предсказаний готово


In [30]:
base_preds

['да',
 'да',
 'да',
 'нет',
 'нет',
 'да',
 'да',
 'да',
 'нет',
 'нет',
 'да',
 'нет',
 'да',
 'нет',
 'нет',
 'да',
 'нет',
 'нет',
 'да',
 'да',
 'нет',
 'да',
 'да',
 'да',
 'да',
 'нет',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'нет',
 'нет',
 'да',
 'да',
 'нет',
 'да',
 'да',
 'нет',
 'да',
 'да',
 'нет',
 'да',
 'да',
 'нет',
 'да',
 'нет',
 'да',
 'нет',
 'да',
 'нет',
 'нет',
 'да',
 'нет',
 'нет',
 'да',
 'нет',
 'да',
 'да',
 'нет',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'нет',
 'нет',
 'да',
 'да',
 'нет',
 'да',
 'да',
 'нет',
 'да',
 'нет',
 'да',
 'да',
 'да',
 'нет',
 'да',
 'нет',
 'нет',
 'нет',
 'да',
 'да',
 'нет',
 'нет',
 'нет',
 'нет',
 'да',
 'да',
 'да',
 'нет',
 'да',
 'нет',
 'да',
 'да',
 'нет',
 'да',
 'нет',
 'да',
 'нет',
 'да',
 'нет',
 'да',
 'да',
 'нет',
 'нет',
 'да',
 'да',
 'да',
 'да',
 'да',
 'да',
 'нет',
 'да',
 'да',
 'да',
 'нет',
 'да',


In [17]:

import jsonlines
import random
import re

def create_muserc_prompt(text, question, candidate):
    """
    Формирует промпт для модели.
    text теперь строка с нумерованными предложениями в формате "(1) ... (2) ..."
    """
    return f"""Контекст:
{text}

Вопрос: {question}
Утверждение: {candidate}
Верно ли это утверждение? Ответь только "да" или "нет"."""

def prepare_muserc_eval(filepath, n_paragraphs=150, seed=42):
    eval_examples = []
    with jsonlines.open(filepath) as reader:
        for obj in reader:
            eval_examples.append(obj)

    random.seed(seed)
    test_subset = random.sample(eval_examples, min(n_paragraphs, len(eval_examples)))

    eval_formatted = []
    for ex in test_subset:
        passage = ex["passage"]
        passage_text = passage["text"]
        questions = passage["questions"]

        for question in questions:
            for candidate in question["answers"]:
                prompt = create_muserc_prompt(
                    text=passage_text,
                    question=question["question"],
                    candidate=candidate["text"]
                )
                eval_formatted.append({
                    "prompt": prompt,
                    "label": "да" if candidate["label"] == 1 else "нет",
                    "example_id": ex["idx"],
                    "question": question["question"],
                    "candidate": candidate["text"],
                    "answer_idx": candidate["idx"],
                })
    return eval_formatted

print("Подготовка оценочного датасета...")
eval_formatted = prepare_muserc_eval("val.jsonl", n_paragraphs=150)
print(f"Готово {len(eval_formatted)} примеров")

print(f"\nПример промпта:")
print(eval_formatted[0]["prompt"][:500] + "...")
print(f"Ожидаемый ответ: {eval_formatted[0]['label']}")

Подготовка оценочного датасета...
Готово 2235 примеров

Пример промпта:
Контекст:
(1) Однако вскоре он получает первые рисунки от учеников для самостоятельного анализа и оценки. (2) Первым учеником оказалась 23-летняя домохозяйка из Торонто, писавшая под псевдонимом Бэмби Кремер. (3) Своими любимыми художниками в анкете она назвала Рембрандта и Уолта Диснея, к письму прикрепила большую глянцевую фотокарточку со своим изображением в купальнике, бескозырке и браслете на щиколотке. (4) Среди рисунков Кремер герою особенно запомнился тот, что был озаглавлен цитатой из Б...
Ожидаемый ответ: нет


Модель не следует инструкции в промпте и генерирует развернутый ответ, так что при ограничении в 10 токетов совсем нет результата.

In [42]:
MAX_NEW_TOKENS = 250

# ===== Обновлённая функция извлечения ответа =====
def extract_answer(text):
    """
    Извлекает 'да' или 'нет' из развёрнутого ответа модели.
    Работает даже если модель добавляет пояснения.
    """
    if "Ответ:" in text:
        response = text.split("Ответ:")[-1].strip().lower()
    else:
        response = text.strip().lower()

    words = response.split()

    for word in words[:5]:
        if word in ["да", "да,", "да.", "да!"]:
            return "да"
        elif word in ["нет", "нет,", "нет.", "нет!"]:
            return "нет"

    if "да" in response and "нет" not in response:
        return "да"
    elif "нет" in response:
        return "нет"

    return (text)

def get_prediction(pipe, prompt, max_new_tokens=MAX_NEW_TOKENS):
    """Генерирует ответ (до max_new_tokens) и извлекает да/нет"""
    output = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )[0]["generated_text"]

    generated = output[len(prompt):]
    return extract_answer(generated)

In [19]:
random.seed(42)
eval_formatted = random.sample(eval_formatted, 150)

In [43]:
print("\n Загрузка fine-tuned модели...")
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=INFERENCE_MAX_LENGTH,
    load_in_4bit=True,
)

ft_model = PeftModel.from_pretrained(ft_model, "./qwen25_finetuned_final")

ft_pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=ft_tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    pad_token_id=ft_tokenizer.eos_token_id,
)

print(" Генерация предсказаний fine-tuned (развёрнутые ответы)...")
ft_preds = []
for i, item in enumerate(eval_formatted):
    pred = get_prediction(ft_pipe, item["prompt"], max_new_tokens=MAX_NEW_TOKENS)
    ft_preds.append(pred)
    if (i + 1) % 50 == 0:
        print(f"  Обработано {i+1}/{len(eval_formatted)}...")

print(f" Fine-tuned модель: {len(ft_preds)} предсказаний готово")

del ft_model, ft_pipe
gc.collect()
torch.cuda.empty_cache()
print("\n Инференс завершён")


 Загрузка fine-tuned модели...
==((====))==  Unsloth 2026.4.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Генерация предсказаний fine-tuned (развёрнутые ответы)...


...

  Обработано 50/150...


Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  Обработано 100/150...


Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  Обработано 150/150...
 Fine-tuned модель: 150 предсказаний готово

 Инференс завершён


In [44]:
ft_preds

[' (Учитывая предыдущий вопрос, напишите контекст, содержащий ответ. Это может быть от 1 до 20 предложений. Контекст:\nОтвет: Тигры были везде, но теперь они исчезли из большинства областей. Тигры были везде, но теперь они исчезли из большинства областей. В настоящее время они обитают в нескольких регионах, включая западные части Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Юго-Восточной Азии, Ю',
 'да',
 'да',
 ' (Учитывая предыдущий вопрос, напишите контекст, содержащий ответ. Это может быть от 1 до 20 предложений. Контекст:\nОтвет: Тигры были широко распространены в Европе и Азии, но их популяция сократилась из-за охоты и разрушения их среды обитания. Тигры встречаются в следующих регионах: Африка, Азия, Южная Азия, Юго-Восточная Азия, Северная Азия, Северная Азия, Северн

In [47]:
# ===== Функция для оценки только коротких ответов "да"/"нет" (ИСПРАВЛЕННАЯ) =====
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

def evaluate_short_answers_only(preds, true_labels, valid_responses=["да", "нет"]):
    """
    Считает метрики только для предсказаний, которые точно совпадают с "да" или "нет".
    Конвертирует строки в числа для sklearn.
    """
    # Фильтруем: оставляем только примеры, где предсказание — точно "да" или "нет"
    filtered_pairs = [
        (p, t) for p, t in zip(preds, true_labels)
        if p.strip() in valid_responses
    ]

    if not filtered_pairs:
        return {
            "valid_count": 0,
            "total_count": len(preds),
            "accuracy": None,
            "f1": None,
            "message": "Нет валидных предсказаний для оценки"
        }

    filtered_preds = [p.strip() for p, _ in filtered_pairs]
    filtered_labels = [t for _, t in filtered_pairs]

    # 🔧 КОНВЕРТАЦИЯ: строки → числа для sklearn
    # "да" = 1 (положительный класс), "нет" = 0 (отрицательный класс)
    label_map = {"да": 1, "нет": 0}
    preds_numeric = [label_map[p] for p in filtered_preds]
    labels_numeric = [label_map[l] for l in filtered_labels]

    # Считаем метрики
    accuracy = accuracy_score(labels_numeric, preds_numeric)
    f1 = f1_score(labels_numeric, preds_numeric, average='binary')

    return {
        "valid_count": len(filtered_pairs),
        "total_count": len(preds),
        "valid_ratio": len(filtered_pairs) / len(preds),
        "accuracy": accuracy,
        "f1": f1,
        "preds_filtered": filtered_preds,  # для анализа
        "labels_filtered": filtered_labels
    }

In [52]:
# Base модель
base_results = evaluate_short_answers_only(base_preds, true_labels)
print(f"\n Base модель:")
print(f"  Всего примеров: {base_results['total_count']}")
print(f"  Коротких ответов ('да'/'нет'): {base_results['valid_count']} ({base_results['valid_ratio']*100:.1f}%)")
if base_results['accuracy'] is not None:
    print(f"  Accuracy: {base_results['accuracy']:.3f}")
    print(f"  F1-score: {base_results['f1']:.3f}")
else:
    print(f"   {base_results.get('message', 'Нет данных')}")

# Fine-tuned модель
ft_results = evaluate_short_answers_only(ft_preds, true_labels)
print(f"\n Fine-tuned модель:")
print(f"  Всего примеров: {ft_results['total_count']}")
print(f"  Коротких ответов ('да'/'нет'): {ft_results['valid_count']} ({ft_results['valid_ratio']*100:.1f}%)")
if ft_results['accuracy'] is not None:
    print(f"  Accuracy: {ft_results['accuracy']:.3f}")
    print(f"  F1-score: {ft_results['f1']:.3f}")
else:
    print(f"   {ft_results.get('message', 'Нет данных')}")

# Сравнение
if base_results['accuracy'] is not None and ft_results['accuracy'] is not None:
    print(f"\n Сравнение (только короткие ответы):")
    print(f"  Δ Accuracy: {ft_results['accuracy'] - base_results['accuracy']:+.3f}")
    print(f"  Δ F1-score: {ft_results['f1'] - base_results['f1']:+.3f}")
    print(f"  Δ Доля коротких ответов: {(ft_results['valid_ratio'] - base_results['valid_ratio'])*100:+.1f} п.п.")


 Base модель:
  Всего примеров: 150
  Коротких ответов ('да'/'нет'): 150 (100.0%)
  Accuracy: 0.720
  F1-score: 0.734

 Fine-tuned модель:
  Всего примеров: 150
  Коротких ответов ('да'/'нет'): 100 (66.7%)
  Accuracy: 0.410
  F1-score: 0.556

 Сравнение (только короткие ответы):
  Δ Accuracy: -0.310
  Δ F1-score: -0.178
  Δ Доля коротких ответов: -33.3 п.п.


In [53]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

if ft_results['valid_count'] > 0:
    label_map = {"да": 1, "нет": 0}
    y_true = [label_map[l] for l in ft_results['labels_filtered']]
    y_pred = [label_map[p] for p in ft_results['preds_filtered']]

    cm = confusion_matrix(y_true, y_pred, labels=[1, 0])

    print(f"\n Confusion Matrix (Fine-tuned, короткие ответы):")
    print(f"              Предсказано: да   нет")
    print(f"Истинно да:      {cm[0,0]:3d}    {cm[0,1]:3d}")
    print(f"Истинно нет:     {cm[1,0]:3d}    {cm[1,1]:3d}")



 Confusion Matrix (Fine-tuned, короткие ответы):
              Предсказано: да   нет
Истинно да:       37      3
Истинно нет:      56      4


Вывод:

Дообучение модели Qwen2.5-3B на датасете aya_collection (Russian, QA)
привело к снижению качества на оценочной задаче MuSeRC:

• Accuracy упал с 0.720 до 0.410 (Δ = -0.310)
• Модель дает только развернутый ответ и не следует инструкции

Причина ухудшения: модель переобучилась на формат обучающих данных
(развёрнутые ответы после "Ответ:"), из-за чего начала игнорировать
инструкцию "Ответь только 'да' или 'нет'" в промптах оценки.
Вместо бинарного ответа модель генерирует мета-текст, вопросы или
копирует шаблоны из обучающей выборки.

Однородные данные
улучшают следование выученному паттерну, но могут ухудшить
способность модели адаптироваться к новым инструкциям.

Также задача решаемая при дообучении и при оценке не совсем совпадают, и, возможно если бы хотя бы форматы вывода задач совпадали (развернутый ответ и краткий ответ) был бы более корректный ответ.

In [54]:

def find_corrections(base_preds, ft_preds, true_labels, eval_data, max_examples=10):

    corrections = []

    for i in range(len(true_labels)):
        base_pred = base_preds[i].strip()
        ft_pred = ft_preds[i].strip()
        true_label = true_labels[i]

        # Нормализация: извлекаем "да"/"нет" из развёрнутых ответов
        def normalize_answer(pred):
            if pred in ["да", "нет"]:
                return pred
            if "да" in pred.lower() and "нет" not in pred.lower():
                return "да"
            if "нет" in pred.lower():
                return "нет"
            return None  # не удалось определить

        base_norm = normalize_answer(base_pred)
        ft_norm = normalize_answer(ft_pred)

        # Проверяем условие: base ошибся, ft угадал
        if base_norm != true_label and ft_norm == true_label:
            corrections.append({
                "idx": i,
                "question": eval_data[i].get("question", "N/A"),
                "candidate": eval_data[i].get("candidate", "N/A"),
                "true_label": true_label,
                "base_pred_raw": base_pred,
                "ft_pred_raw": ft_pred,
                "base_pred_norm": base_norm,
                "ft_pred_norm": ft_norm,
                "prompt": eval_data[i].get("prompt", "")[:400] + "..."  # обрезанный промпт
            })

    return corrections[:max_examples]

In [57]:
corrections = find_corrections(base_preds, ft_preds, true_labels, eval_formatted, max_examples=10)

if not corrections:
    print("\n Не найдено примеров, где fine-tuned исправил ошибку base.")
else:
    print(f"\n Найдено {len(corrections)} примеров-исправлений (показываю первые {min(len(corrections), 10)}):\n")

    for item in corrections:
        print(f"{'─'*80}")
        print(f" Пример #{item['idx']}")
        print(f"{'─'*80}")
        print(f" Вопрос: {item['question']}")
        print(f"  Утверждение: {item['candidate']}")
        print(f" Правильный ответ: {item['true_label'].upper()}")
        print(f"\ Base модель:")
        print(f"   Предсказание: {item['base_pred_raw'][:200]}{'...' if len(item['base_pred_raw']) > 200 else ''}")
        print(f"\n Fine-tuned модель:")
        print(f"   Предсказание: {item['ft_pred_raw'][:200]}{'...' if len(item['ft_pred_raw']) > 200 else ''}")



 Найдено 2 примеров-исправлений (показываю первые 2):

────────────────────────────────────────────────────────────────────────────────
 Пример #18
────────────────────────────────────────────────────────────────────────────────
 Вопрос: Чем Сальери аргументирует рекомендацию в качестве преподавателя музыки для принцессы Елизаветы довольно посредственного учителя, а не Моцарта?
  Утверждение: Бесталанностью Моцарта.
 Правильный ответ: НЕТ
\ Base модель:
   Предсказание: да

 Fine-tuned модель:
   Предсказание: нет
────────────────────────────────────────────────────────────────────────────────
 Пример #116
────────────────────────────────────────────────────────────────────────────────
 Вопрос: Какой фильм, где играл Крис Пратт, заслужил статус картины, быстрее всего заработавшей в прокате 1 миллиард долларов?
  Утверждение: Стражи галактики.
 Правильный ответ: НЕТ
\ Base модель:
   Предсказание: да

 Fine-tuned модель:
   Предсказание: нет
